In [ ]:
%%capture
! pip install -U evaluate transformers accelerate

In [ ]:
import pandas as pd
import numpy as np

# Tokenization utilities
from transformers import (AutoTokenizer,
                          AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup,
                          Trainer,
                          TrainingArguments)

from huggingface_hub import login

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import random_split, DataLoader

# Sklearn
import sklearn
from sklearn.utils.class_weight import compute_class_weight

# Evaluate
import evaluate

# Runtime utilities
import math
import time
from tqdm import tqdm, trange
import json

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# import os
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
login(token='')

In [ ]:
class ModelFinetuner:
    """
    A class for fine-tuning the DepRoBERTa model on a social media dataset for stigma detection task.
    """

    def __init__(self, file_path):
        """
        Initialize the ModelFinetuner class.

        Args:
            file_path (str): The path to the JSON file containing the dataset.
        """
        self.file_path = file_path

        self.accuracy = evaluate.load('accuracy')
        self.precision = evaluate.load('precision')
        self.recall = evaluate.load('recall')
        self.f1 = evaluate.load('f1')

    def load_dataset(self):
        """
        Load the dataset from the JSON file.
        """
        with open(f'{self.file_path}/train_data.json', 'r') as f:
          self.train_data = json.loads(f.read())
          f.close()

        with open(f'{self.file_path}/val_data.json', 'r') as f:
          self.val_data = json.loads(f.read())
          f.close()


    def create_dataset(self):
        """
        Split the dataset into train, validation, and test sets.

        Args:
            test_size (float): The proportion of the dataset to include in the test split.
            val_size (float): The proportion of the dataset to include in the validation split.
        """

        self.load_dataset()

        train_txt, train_label = self.format_dataset(self.train_data)
        val_txt, val_label = self.format_dataset(self.val_data)

        self.tokenizer = AutoTokenizer.from_pretrained("rafalposwiata/deproberta-large-depression", do_lower_case=True)

        train_encodings = self.tokenizer(train_txt, truncation=True, padding=True)
        val_encodings = self.tokenizer(val_txt, truncation=True, padding=True)

        self.train_dataset = PostDataset(train_encodings, train_label)
        self.val_dataset = PostDataset(val_encodings, val_label)


    def format_dataset(self, data):
        """
        Create a PyTorch dataset from the given encodings and labels.

        Args:
            encodings (dict): The tokenized input encodings.
            labels (list): The corresponding labels.

        Returns:
            IMDbDataset: A PyTorch dataset object.
        """
        sentences = []
        labels = []

        for post in data:
          title, post_text = post['title'], post['post text']
          if title is None:
            title = ''
          if post_text is None:
            post_text = ''

          if post['label'] == 'no stigma' or post['label'] == 'unclear':
            labels.append(0)
            sentences.append(f'{title}-{post_text}')
          elif post['label'] == 'yes stigma':
            labels.append(1)
            sentences.append(f'{title}-{post_text}')

        return (sentences, labels)


    def get_class_weight(self):
      _, train_label = self.format_dataset(self.train_data)
      classes = np.unique(train_label)

      class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=train_label)

      # Convert to torch tensor
      class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
      return class_weights_tensor


    def fine_tune(self, epochs=5, batch_size=16, warmup_steps=500, weight_decay=0.01):
        """
        Fine-tune the DepRoBERTa model on the training data.

        Args:
            epochs (int): The number of training epochs.
            batch_size (int): The batch size for training.
            warmup_steps (int): The number of warmup steps for the learning rate scheduler.
            weight_decay (float): The strength of weight decay regularization.
        """
        training_args = TrainingArguments(
            output_dir='./results',
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            warmup_steps=warmup_steps,
            weight_decay=weight_decay,
            logging_dir='./logs',
            logging_steps=10,
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model='f1',
            report_to="none"
            )

        self.model = AutoModelForSequenceClassification.from_pretrained(
            "rafalposwiata/deproberta-large-depression",
            output_attentions = False,
            output_hidden_states = False,
            )

        self.model.classifier.out_proj = nn.Linear(1024, 2)
        self.model.num_labels = 2

        for param in self.model.base_model.parameters():
            param.requires_grad = False

        # Unfreeze the classification head
        for param in self.model.classifier.parameters():
            param.requires_grad = True

        self.model.cuda()

        self.trainer = WeightedLossTrainer(
            model=self.model,
            args=training_args,
            train_dataset=self.train_dataset,
            eval_dataset=self.val_dataset,
            compute_metrics=self.compute_metrics,
        )

        self.trainer.train()


    def compute_metrics(self, pred):
        """
        Compute evaluation metrics based on the predictions.

        Args:
            pred (EvalPrediction): The model's predictions.

        Returns:
            dict: A dictionary containing the computed metrics.
        """
        predictions, labels = pred
        predictions = np.argmax(predictions, axis=1)
        result = {'accuracy': self.accuracy.compute(predictions=predictions, references=labels)['accuracy'],
              'precision': self.precision.compute(predictions=predictions, references=labels, average = 'macro')['precision'],
              'recall': self.recall.compute(predictions=predictions, references=labels, average = 'macro')['recall'],
              'f1': self.f1.compute(predictions=predictions, references=labels, average = 'macro')['f1']}
        return result

    def evaluate_model(self):
        """
        Evaluate the fine-tuned model on the test set.
        """
        print(self.trainer.evaluate(self.test_dataset))

    def save_model(self, model_name):
        """
        Save the fine-tuned model and tokenizer to the Hugging Face Hub.

        Args:
            model_name (str): The name of the model on the Hugging Face Hub.
        """
        repo_name = f'haji80mr-uoft/{model_name}'
        self.tokenizer.push_to_hub(repo_name)
        self.model.push_to_hub(repo_name)



class PostDataset(torch.utils.data.Dataset):
    """
    A PyTorch dataset for the movie genre classification task.
    """

    def __init__(self, encodings, labels):
        """
        Initialize the IMDbDataset class.

        Args:
            encodings (dict): The tokenized input encodings.
            labels (list): The corresponding labels.
        """
        # TODO: Implement initialization logic

        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        """
        Get a single item from the dataset.

        Args:
            idx (int): The index of the item to retrieve.

        Returns:
            dict: A dictionary containing the input encodings and labels.
        """
        # TODO: Implement item retrieval logic
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        """
        Get the length of the dataset.

        Returns:
            int: The number of items in the dataset.
        """
        # TODO: Implement length computation logic
        return len(self.labels)

In [ ]:
finetuner = ModelFinetuner('/content/drive/MyDrive/stigma_project/final experiment results')
finetuner.load_dataset()

class_weights = finetuner.get_class_weight()
loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(device))

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, num_items_in_batch = 8, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
finetuner.create_dataset()

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

In [ ]:
finetuner.fine_tune()

config.json:   0%|          | 0.00/924 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.724300,0.667025,0.774834,0.503194,0.507801,0.488238
2,0.608100,0.524087,0.933775,0.725862,0.639362,0.670017
3,0.524300,0.431934,0.920530,0.699687,0.771631,0.728417
4,0.485600,0.412483,0.814570,0.609100,0.807801,0.627555
5,0.665400,0.406711,0.761589,0.586064,0.779433,0.581846


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [ ]:
finetuner.trainer.evaluate(finetuner.val_dataset)

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


{'eval_loss': 0.431934118270874,
 'eval_accuracy': 0.9205298013245033,
 'eval_precision': 0.6996871741397289,
 'eval_recall': 0.7716312056737589,
 'eval_f1': 0.7284172661870504,
 'eval_runtime': 12.8662,
 'eval_samples_per_second': 11.736,
 'eval_steps_per_second': 0.777,
 'epoch': 5.0}

In [ ]:
with open(f'{finetuner.file_path}/test_data.json', 'r') as f:
          test_data = json.loads(f.read())
          f.close()

test_txt, test_label = finetuner.format_dataset(test_data)

test_encodings = finetuner.tokenizer(test_txt, truncation=True, padding=True)

test_dataset = PostDataset(test_encodings, test_label)

In [ ]:
finetuner.trainer.evaluate(test_dataset)

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


{'eval_loss': 0.5578632950782776,
 'eval_accuracy': 0.9039735099337748,
 'eval_precision': 0.6840366039654295,
 'eval_recall': 0.6513377926421404,
 'eval_f1': 0.6654570457236717,
 'eval_runtime': 26.7121,
 'eval_samples_per_second': 11.306,
 'eval_steps_per_second': 0.711,
 'epoch': 5.0}

In [ ]:
finetuner.save_model('DepRoberta_finetuned_V0')

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /tmp/tmplaj40q1n/model.safetensors    :   0%|          |  552kB / 1.42GB            

# Getting the output logits of Fine-tuned model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('haji80mr-uoft/DepRoberta_finetuned_V1_kaggle')
model = AutoModelForSequenceClassification.from_pretrained('haji80mr-uoft/DepRoberta_finetuned_V1_kaggle', num_labels=2)
model.to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/957 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/868 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 1024, padding_idx=1)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=Tru

In [ ]:
with open('/content/drive/MyDrive/stigma_project/semi-supervised learning/integrated.json', 'r') as f:
  all_data = json.loads(f.read())
  f.close()

In [ ]:
all_sent = []

for d in all_data:
  all_sent.append(d['input text'])

In [ ]:
all_logits = []

for sent in tqdm(all_sent):
  encode = tokenizer.encode_plus(sent, truncation=True, padding='max_length', return_tensors='pt')
  encode = {k: v.to(device) for k, v in encode.items()}

  ans = model(**encode)
  logit = ans.logits.cpu().detach().tolist()[0]
  all_logits.append(logit)

100%|██████████| 1386/1386 [02:17<00:00, 10.08it/s]


In [ ]:
with open('/content/drive/MyDrive/stigma_project/semi-supervised learning/all_logits_Deproberta.json', 'w') as f:
  f.write(json.dumps(all_logits))
  f.close()